# RAG Retrieval Debugging With Kayak

Treat this as the smallest possible debugging session.

Start from one annoying situation:

- the right document is in the corpus, but retrieval still puts something else on top

This notebook is intentionally narrow:

- one toy setup
- one concrete failure mode
- one visible ranking change when the simplification is removed

Backstage verification note: the core claims from this notebook are covered by `python.tests.test_course_rag_debugging_smoke`.

In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python" / "kayak").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "python"))

import kayak

print("Working from repo root:", REPO_ROOT)
print("Backends available here:", kayak.available_backends())

In [ ]:
DIM = 64
TOKEN_TO_INDEX: dict[str, int] = {}


def token_vector(token: str) -> np.ndarray:
    index = TOKEN_TO_INDEX.setdefault(token, len(TOKEN_TO_INDEX))
    if index >= DIM:
        raise ValueError("Increase DIM for this notebook example.")
    vector = np.zeros(DIM, dtype=np.float32)
    vector[index] = np.float32(1.0)
    return vector


def encode_tokens(tokens: list[str]) -> np.ndarray:
    return np.stack([token_vector(token) for token in tokens])


def dense_mean(tokens: list[str]) -> np.ndarray:
    return encode_tokens(tokens).mean(axis=0)


def cosine_similarity(left: np.ndarray, right: np.ndarray) -> float:
    return float(np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right)))


def maxsim_breakdown(query_tokens: list[str], doc_tokens: list[str]) -> list[tuple[str, float]]:
    document_vectors = encode_tokens(doc_tokens)
    rows: list[tuple[str, float]] = []
    for token in query_tokens:
        query_row = token_vector(token)
        best = float(np.max(document_vectors @ query_row))
        rows.append((token, best))
    return rows

In [ ]:
query_tokens = ["cancel", "subscription"]

documents = {
    "doc-relevant": query_tokens + [f"noise-{i}" for i in range(20)],
    "doc-partial": ["cancel", "cancel", "cancel", "cancel"],
    "doc-other": ["billing", "invoice"],
}

index = kayak.documents(
    list(documents.keys()),
    [encode_tokens(tokens) for tokens in documents.values()],
).pack()
query = kayak.query(encode_tokens(query_tokens), text="cancel subscription")

kayak_hits = kayak.search(
    query,
    index,
    k=3,
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)
dense_scores = sorted(
    (
        (doc_id, cosine_similarity(dense_mean(query_tokens), dense_mean(tokens)))
        for doc_id, tokens in documents.items()
    ),
    key=lambda row: row[1],
    reverse=True,
)

print("If I keep full token interaction:")
for hit in kayak_hits:
    print(f"  {hit.doc_id:12s} score={hit.score:.3f}")

print("\nIf I collapse everything to one mean-pooled vector:")
for doc_id, score in dense_scores:
    print(f"  {doc_id:12s} score={score:.3f}")

In this toy setup:

- `doc-relevant` really does contain the full evidence, but it also carries a lot of irrelevant content
- `doc-partial` keeps shouting one topical token over and over
- the mean-pooled baseline gets pulled toward the loud partial overlap
- Kayak exact late interaction keeps the two-token match visible and ranks the relevant document first

This is not a benchmark claim.
It is the smallest clean example of a miss many people already recognize from real systems.

In [ ]:
for doc_id, tokens in documents.items():
    breakdown = maxsim_breakdown(query_tokens, tokens)
    total = sum(score for _, score in breakdown)
    print(f"Looking inside {doc_id}:")
    for token, score in breakdown:
        print(f"  best match for query token {token:12s} = {score:.1f}")
    print(f"  total maxsim score = {total:.1f}\n")

In [ ]:
batch_queries = {
    "cancel subscription": ["cancel", "subscription"],
    "cancel only": ["cancel"],
    "invoice": ["invoice"],
}

query_batch = kayak.query_batch(
    [encode_tokens(tokens) for tokens in batch_queries.values()],
)

loop_hits = tuple(
    kayak.search(
        kayak.query(encode_tokens(tokens), text=name),
        index,
        k=1,
        backend=kayak.NUMPY_REFERENCE_BACKEND,
    )
    for name, tokens in batch_queries.items()
)
batch_hits = kayak.search_batch(
    query_batch,
    index,
    k=1,
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)

assert batch_hits == loop_hits

for name, hits in zip(batch_queries, batch_hits, strict=True):
    print(name, "still resolves to ->", hits[0].doc_id)

## Boundary

What this notebook lets the learner say honestly:

- token-level interaction is a useful debugging lens
- the basic Kayak primitives make that comparison explicit
- exact local search is a good first reference path

What it does not let the learner say honestly:

- Kayak wins every benchmark
- mean pooling is always wrong
- this toy already settles production behavior

That stronger story has to come later from judged slices and benchmark notes.